# 03 — Modeling and Evaluation
**Goal:** compare interpretable baseline classifiers for known first-stage landing outcomes.

The model uses only a small metadata feature set as a starting point. Validate feature timing and leakage assumptions before treating this as a launch-time predictor.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / "data" / "spacex_launches_raw.csv")
df["landing_success"] = df["landing_success"].map({"True": 1, "False": 0, True: 1, False: 0})
df = df[df["landing_success"].notna() & (df["upcoming"] != True)].copy()
df["landing_success"] = df["landing_success"].astype(int)
print(df["landing_success"].value_counts())

In [ ]:
# Minimal baseline features. Flight number/year may encode time; explicitly discuss this choice.
numeric_features = ["flight_number", "payload_count", "payload_mass_kg", "launch_year", "launch_month"]
categorical_features = ["rocket_name", "launchpad_name", "orbit", "core_reused"]

for col in numeric_features:
    if col not in df:
        df[col] = np.nan
for col in categorical_features:
    if col not in df:
        df[col] = "Unknown"

X = df[numeric_features + categorical_features].copy()
y = df["landing_success"]
# Keep a stratified split where possible.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced", min_samples_leaf=3),
    "SVM": SVC(probability=True, class_weight="balanced", random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}
rows = []
fitted = {}
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    row = {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0)
    }
    try:
        row["roc_auc"] = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    except Exception:
        row["roc_auc"] = np.nan
    rows.append(row)
    fitted[name] = pipe
results = pd.DataFrame(rows).sort_values("f1", ascending=False)
display(results.round(3))
(ROOT / "outputs").mkdir(exist_ok=True)
results.to_csv(ROOT / "outputs" / "model_metrics.csv", index=False)

In [ ]:
# Inspect the top model's test-set errors
best_name = results.iloc[0]["model"]
best_model = fitted[best_name]
pred = best_model.predict(X_test)
print("Selected by test F1 for this exploratory baseline:", best_name)
print(classification_report(y_test, pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["No landing", "Landing"])
plt.title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.savefig(ROOT / "outputs" / "confusion_matrix.png", dpi=160)
plt.show()

## Required reporting checklist
- Report the held-out sample size and class distribution.
- Include the metric table and confusion matrix produced by your run.
- Compare precision and recall; discuss false positives and false negatives.
- Tune hyperparameters using cross-validation on training data only if you extend this notebook.
- Consider a chronological split and a leakage audit before making claims about deployment performance.
- Do not claim the best model until the notebook has been run successfully and the test-set metrics have been reviewed.